In [0]:
%pip install google-genai youtube-transcript-api google-api-python-client
dbutils.library.restartPython()

In [0]:
import datetime
from googleapiclient.discovery import build
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

# Setup YouTube API
YOUTUBE_API_KEY = dbutils.secrets.get(scope="courseify", key="youtube-api")
youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)

COURSE_ID = "py_eng" 
CHANNEL_NAME = "JennyslecturesCSIT"
PLAYLIST_ID = "PLdo5W4Nhv31bZSiqiOL5ta39vSnBxpOPT" 

def get_playlist_videos(playlist_id):
    videos = []
    next_page_token = None
    while True:
        request = youtube.playlistItems().list(part="snippet", playlistId=playlist_id, maxResults=50, pageToken=next_page_token)
        response = request.execute()
        for item in response['items']:
            title = item['snippet']['title']
            if title not in ["Private video", "Deleted video"]:
                videos.append({
                    "video_id": item['snippet']['resourceId']['videoId'],
                    "title": title
                })
        next_page_token = response.get('nextPageToken')
        if not next_page_token: break
    return videos

def ingest_to_bronze():
    print(f"Fetching raw videos for {COURSE_ID}...")
    videos = get_playlist_videos(PLAYLIST_ID)
    
    data_to_insert = []
    current_time = datetime.datetime.now()
    
    # Get existing video IDs in Bronze to avoid duplicates
    existing_df = spark.sql(f"SELECT video_id FROM courseify.default.course_bronze WHERE course_id = '{COURSE_ID}'")
    existing_vids = [row.video_id for row in existing_df.collect()]
    
    for vid in videos:
        if vid['video_id'] not in existing_vids:
            data_to_insert.append((
                COURSE_ID, vid['video_id'], vid['title'], CHANNEL_NAME, PLAYLIST_ID, current_time
            ))
            
    if data_to_insert:
        schema = StructType([
            StructField("course_id", StringType(), True),
            StructField("video_id", StringType(), True),
            StructField("video_title", StringType(), True),
            StructField("channel_name", StringType(), True),
            StructField("playlist_id", StringType(), True),
            StructField("ingested_at", TimestampType(), True)
        ])
        df = spark.createDataFrame(data_to_insert, schema=schema)
        df.write.format("delta").mode("append").saveAsTable("courseify.default.course_bronze")
        print(f"✅ Job 1 Success! {len(data_to_insert)} new raw videos added to Bronze Table.")
    else:
        print("✅ Bronze Table is already up to date.")

if __name__ == "__main__":
    ingest_to_bronze()

In [0]:
%sql
-- update courseify.default.course_bronze set course_id = "py_eng"

In [0]:
%sql
select * from courseify.default.course_bronze